In [ ]:
import os
from typing import Any, cast

import torch
import torch.nn as nn
from torchvision.models import resnet18
import genesis as gs
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from tensordict.nn import TensorDictModule, TensorDictSequential
from tensordict.nn.distributions import NormalParamExtractor

from tensordict import TensorDict
from torchrl.data import Composite, Bounded

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
    Resize,
)
from torchrl.envs import EnvBase
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator, IndependentNormal
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm


In [2]:
gs.init(backend=gs.cpu, logging_level="warning")

In [ ]:
# gs.destroy()

[Genesis] [20:56:13] [INFO] 💤 Exiting Genesis and caching compiled kernels...


In [ ]:
YOUBOT_DESCRIPTION = r"D:\ros\src\youbot_description"

JOINTS = [
    "wheel_joint_fl",
    "wheel_joint_fr",
    "wheel_joint_bl",
    "wheel_joint_br",
    "arm_joint_1",
    "arm_joint_2",
    "arm_joint_3",
    "arm_joint_4",
    "arm_joint_5",
    "gripper_finger_joint_l",
    "gripper_finger_joint_r",
]

joint2id = {k: i for i, k in enumerate(JOINTS)}

LINKS = [
    "arm_link_5",
    "plate_link",
    "gripper_finger_link_l",
    "gripper_finger_link_r",
]

link2id = {k: i for i, k in enumerate(LINKS)}

BATCH_RENDERER = False

num_joints = len(JOINTS)

device = torch.device("cuda:0")

epoch_num_env_steps = 10

max_steps = 200

In [ ]:
class VecEnv(EnvBase):
    def __init__(self,
                 n_envs: int = 1,
                 device=None,
                 show_viewer: bool = False
                 ):
        super().__init__(batch_size=torch.Size([n_envs]), device=device)
        self.set_device = device
        self.n_envs = n_envs
        self.epoch_num_env_steps = epoch_num_env_steps

        self._set_seed(42)
        self.max_steps = max_steps
        self.steps_count = torch.tensor([0] * self.n_envs, dtype=torch.int, device=self.set_device)
        
        self.scene = gs.Scene(
            show_viewer=show_viewer,
            viewer_options=gs.options.ViewerOptions(max_FPS=120),
            vis_options=gs.options.VisOptions(),
        )
        self.plane = self.scene.add_entity(gs.morphs.Plane())

        self.bot = self.scene.add_entity(
            gs.morphs.URDF(
                file=os.path.join(YOUBOT_DESCRIPTION, "robots/youbot.urdf"),
                pos=(0, 0, 0.1),
                merge_fixed_links=False
            ),
        )

        self.cylinder_height = torch.tensor([0.1], device=self.set_device)
        self.plate_height = 0.025

        self.target_cylinder = self.scene.add_entity(
            gs.morphs.Cylinder(pos=(1.0, 1.0, 0.05), height=self.cylinder_height, radius=0.015),
        )

        self.reward_flags = TensorDict({
            "has_opened": torch.zeros(self.n_envs, dtype=torch.bool, device=self.set_device),
            "has_lifted": torch.zeros(self.n_envs, dtype=torch.bool, device=self.set_device),
        }, batch_size=[self.n_envs], device=self.set_device)

        self.bot_dofs_idx = [self.bot.get_joint(name).dofs_idx_local[0] for name in JOINTS]
        self.bot_links_idx = [self.bot.get_link(name).idx_local for name in LINKS]

        self.force_koeff = torch.as_tensor(
            np.stack([
                self.bot.get_joint(joint)._dofs_force_range[:, -1]
                for joint in JOINTS
            ]),
            device=self.set_device,
            dtype=torch.float32,
        ).squeeze(-1)

        if BATCH_RENDERER:
            renderer_opts = gs.sensors.BatchRendererCameraOptions(
                res=(640, 480),
                pos=(0.05, 0.0, 0.05),  # Offset from link frame
                lookat=(0.25, -0.02, 0.6),  # Look direction
                up=(0.0, 0.0, 1.0),
                entity_idx=self.bot.idx,  # Attach to robot
                link_idx_local=self.bot_links_idx[link2id["arm_link_5"]],  # End-effector link
                lights=[
                    {
                        "pos": (2.0, 2.0, 5.0),
                        "color": (1.0, 1.0, 1.0),
                        "intensity": 1.0,
                        "directional": True,
                        "castshadow": True,
                    }
                ],
            )
        else:
            renderer_opts = gs.sensors.RasterizerCameraOptions(
                res=(640, 480),
                pos=(0.05, 0.0, 0.05),  # Offset from link frame
                lookat=(0.25, -0.02, 0.6),  # Look direction
                up=(0.0, 0.0, 1.0),
                entity_idx=self.bot.idx,  # Attach to robot
                link_idx_local=self.bot_links_idx[link2id["arm_link_5"]],  # End-effector link
                # use_rasterizer=True,
            )

        self.camera = self.scene.add_sensor(cast(Any, renderer_opts))

        self.scene.build(n_envs=self.n_envs, env_spacing=(1.0, 1.0))

        self.set_initial_state()

        self.observation_spec = Composite(
            observation=Bounded(
                low=0, high=255, 
                shape=(self.n_envs, 3, 480, 640), 
                dtype=torch.uint8,
                device=self.set_device
            ),
            shape=(self.n_envs,),
            device=self.set_device
        )

        self.action_spec = Bounded(
            low=-float("inf"), high=float("inf"), 
            shape=(self.n_envs, num_joints),
            dtype=torch.float32,
            device=self.set_device
        )

        self.reward_spec = Bounded(
            low=-float("inf"), high=float("inf"),
            shape=(self.n_envs, 1),
            device=self.set_device
        )
        
        self.done_spec = Composite(
            done=Bounded(
                low=0, high=1,
                shape=(self.n_envs, 1),
                dtype=torch.bool,
                device=self.set_device
                ),
            terminated=Bounded(
                low=0, high=1,
                shape=(self.n_envs, 1),
                dtype=torch.bool,
                device=self.set_device
                ),
            shape=(self.n_envs,),
            device=self.set_device
        )

    def set_initial_state(self, envs_idx=None):
        if envs_idx is None:
            envs_idx = list(range(self.n_envs))

        pos = torch.stack([
            torch.empty(2, device=self.set_device).uniform_(-0.5, 0.5, generator=self.rngs[i])
            for i in range(self.n_envs)
        ])

        pos[:, 0] += 1.0

        pos = torch.cat([pos, (self.cylinder_height / 2).expand(pos.shape[0]).unsqueeze(-1)], dim=-1)
        
        self.target_cylinder.set_pos(pos=pos[envs_idx], envs_idx=envs_idx)

    def get_cylinder_slope(self):
        q = self.target_cylinder.get_quat().to(device=self.set_device)
        q = q / q.norm(dim=-1, keepdim=True)

        q_xyz = q[:, :3]
        q_w = q[:, -1].unsqueeze(-1)

        v0 = torch.tensor([1.0, 0.0, 0.0], device=self.set_device)
        v0 = v0.expand_as(q_xyz)

        t = 2 * torch.cross(q_xyz, v0, dim=-1)
        v = v0 + q_w * t + torch.cross(q_xyz, t, dim=-1)

        v = v / v.norm(dim=-1, keepdim=True)

        dot = (v * v0).sum(dim=-1).clamp(-1.0, 1.0)
        theta = torch.acos(dot)

        theta_deg = torch.rad2deg(theta)

        return theta_deg

    def get_gripper_state(self):
        bots_links_pos, _ = self.get_main_poses()
        l_finger_pose = bots_links_pos[:, link2id["gripper_finger_link_l"]]
        r_finger_pose = bots_links_pos[:, link2id["gripper_finger_link_r"]]
        l_anchor_pose = self.bot.get_joint("gripper_finger_joint_l").get_anchor_pos().to(self.set_device)
        r_anchor_pose = self.bot.get_joint("gripper_finger_joint_r").get_anchor_pos().to(self.set_device)

        l_distance = torch.sum((l_finger_pose - l_anchor_pose) ** 2, -1) ** 0.5
        r_distance = torch.sum((r_finger_pose - r_anchor_pose) ** 2, -1) ** 0.5

        alpha_closed = 0.001
        alpha_opened = 0.012

        return TensorDict({
            "l_closed": torch.round(l_distance, decimals=3) <= alpha_closed,
            "r_closed": torch.round(r_distance, decimals=3) <= alpha_closed,
            "l_opened": torch.round(l_distance, decimals=3) >= alpha_opened,
            "r_opened": torch.round(r_distance, decimals=3) >= alpha_opened,
        }, batch_size=bots_links_pos.shape[0], device=self.set_device)

    def get_main_poses(self):
        bots_links_pos = self.bot.get_links_pos(self.bot_links_idx)
        cylinder_pos = self.target_cylinder.get_pos()

        return bots_links_pos.to(self.set_device), cylinder_pos.to(self.set_device)

    def _get_observation(self):
        images = self.camera.read()
        return torch.tensor(images.rgb, dtype=torch.uint8, device=self.set_device).permute(0, 3, 1, 2)
    
    def _compute_reward(self, tensordict):
        bots_links_pos, cylinder_pos = self.get_main_poses()
        hands_pos = bots_links_pos[:, link2id["arm_link_5"]]

        distance = torch.norm(hands_pos - cylinder_pos, dim=-1, keepdim=True)
        total_reward = -distance

        prev_action = tensordict["prev_action"]
        action = tensordict["action"]

        beta_action = 0.5
        total_reward -= torch.sum(
            (action - prev_action) ** 2,
            dim=-1,
            keepdim=True
        ) * beta_action

        gripper_state = self.get_gripper_state()

        has_opened = (
            gripper_state["l_opened"]
            & gripper_state["r_opened"]
            & (~self.reward_flags["has_opened"])
        )

        total_reward += has_opened.unsqueeze(-1).float() * 5.0
        self.reward_flags["has_opened"] |= has_opened

        cylinder_slope = self.get_cylinder_slope()

        alpha_slope = 15.0
        beta_slope = 0.2

        penalty = (
            (cylinder_slope - alpha_slope)
            .clamp(min=0.0)
            .unsqueeze(-1)
            * beta_slope
        )

        total_reward -= penalty

        cylinder_pos[:, -1] -= self.cylinder_height / 2

        alpha_lifted = 0.05

        lifted = (
            (cylinder_pos[:, -1] >= alpha_lifted)
            & (~self.reward_flags["has_lifted"])
        )

        total_reward += lifted.unsqueeze(-1).float() * 5.0
        self.reward_flags["has_lifted"] |= lifted

        plate_pos = bots_links_pos[:, link2id["plate_link"]]
        plate_pos[:, -1] += self.plate_height

        in_air = cylinder_pos[:, -1] >= alpha_lifted

        distance = torch.norm(
            plate_pos - cylinder_pos,
            dim=-1,
            keepdim=True
        )

        total_reward += in_air.unsqueeze(-1).float() * (1.0 - distance)

        return total_reward
    
    def _is_terminated(self):
        bots_links_pos, cylinder_pos = self.get_main_poses()
        plate_pos = bots_links_pos[:, link2id["plate_link"]]

        plate_pos[:, -1] += self.plate_height
        cylinder_pos[:, -1] -= self.cylinder_height / 2
        cylinder_angle_error = self.get_cylinder_slope()

        horizontal_distance = torch.sum((plate_pos[:, :-1] - cylinder_pos[:, :-1]) ** 2, -1) ** 0.5
        vertical_distance = torch.abs(plate_pos[:, -1] - cylinder_pos[:, -1])

        horizontal_alpha = 0.12
        vertical_alpha = 0.005 
        angle_alpha = 2.0

        return ((horizontal_distance <= horizontal_alpha)
                & (vertical_distance <= vertical_alpha)
                & (cylinder_angle_error <= angle_alpha))

    def _reset(self, tensordict=None):
        if tensordict is not None:
            done = tensordict["done"].squeeze(-1)
            idx = done.nonzero(as_tuple=True)[0]

            prev_action = tensordict["prev_action"].clone()

            if idx.numel() > 0:
                prev_action[idx] = 0.0

                self.scene.reset(envs_idx=idx.cpu().numpy())
                self.set_initial_state(envs_idx=idx.cpu().numpy())

                self.reward_flags["has_opened"][idx] = False
                self.reward_flags["has_lifted"][idx] = False
        else:
            done = torch.zeros(
                (self.n_envs,),
                dtype=torch.bool,
                device=self.set_device
            )

            prev_action = torch.zeros(
                (self.n_envs, num_joints),
                device=self.set_device
            )

        obs = self._get_observation()

        done = done.unsqueeze(-1)

        return TensorDict(
            {
                "observation": obs,
                "done": done,
                "terminated": done,
                "prev_action": prev_action,
            },
            batch_size=self.batch_size,
            device=self.set_device,
        )
    
    def _step(self, tensordict):
        action = tensordict["action"] * self.force_koeff
        
        self.bot.control_dofs_velocity(
            velocity=action.squeeze().cpu().numpy(),
            dofs_idx_local=self.bot_dofs_idx,
        )

        for _ in range(self.epoch_num_env_steps):
            self.scene.step()

        obs = self._get_observation()
        reward = self._compute_reward(tensordict)
        terminated = self._is_terminated()

        self.steps_count += 1

        done = terminated | (self.steps_count >= self.max_steps)

        out_tensordict = tensordict.clone()

        if torch.any(done):
            reset = self._reset(tensordict)

            out_tensordict["next"] = {
                "observation": reset["observation"],
                "reward": reward,
                "done": reset["done"],
                "terminated": reset["terminated"],
                "prev_action": reset["prev_action"]
            }
        else:
            out_tensordict["next"] = {
                "observation": obs,
                "reward": reward,
                "done": done,
                "terminated": terminated,
                "prev_action": out_tensordict["action"]
            }

        out_tensordict["reward"] = reward

        return out_tensordict

    def _set_seed(self, seed: int):
        self.np_random = np.random.default_rng(seed)
        self.rngs = []
        for i in range(self.n_envs):
            g = torch.Generator(device=self.set_device)
            g.manual_seed(seed + i)
            self.rngs.append(g)

In [ ]:
env = VecEnv(n_envs=4, device=device)
transformed_env = TransformedEnv(
    env,
    Resize(320, 320, in_keys="observation", out_keys="observation", interpolation="bilinear")
)

In [20]:
frames_per_batch = 64
total_frames = 100_000

num_epochs = 5
sub_batch_size = 8

lr = 1e-4
gamma = 0.995
lmbda = 0.97

clip_epsilon = 0.1
entropy_eps = 1e-3

max_grad_norm = 0.5

action_dim = num_joints

In [17]:
class ResNetEncoder(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()

        self.backbone = resnet18(weights=None if not pretrained else "IMAGENET1K_V1", norm_layer=self.gn)

        self.backbone.maxpool = nn.Identity()

        self.features = nn.Sequential(*list(self.backbone.children())[:-1])
        # [B, 512, 1, 1]

        self.flatten = nn.Flatten()

    def gn(self, num_channels):
        return nn.GroupNorm(num_groups=32, num_channels=num_channels)

    def forward(self, x):
        # x: [B, 3, H, W]
        x = x / 255.0
        x = self.features(x)
        x = self.flatten(x)  # [B, 512]
        return x

In [ ]:
encoder_net = ResNetEncoder().to(device=device)

actor_net = nn.Sequential(
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 2 * action_dim),
).to(device=device)

critic_net = nn.Sequential(
    nn.Linear(512, 256),
    nn.Tanh(),
    nn.Linear(256, 256),
    nn.Tanh(),
    nn.Linear(256, 1),
).to(device=device)

feature_extractor = TensorDictModule(
    encoder_net, in_keys=["observation"], out_keys=["hidden"]
)

actor_head = TensorDictModule(
    actor_net, in_keys=["hidden"], out_keys=["loc_scale"]
)

extractor = TensorDictModule(
    NormalParamExtractor(), in_keys=["loc_scale"], out_keys=["loc", "scale"]
)

value_head = TensorDictModule(
    critic_net, in_keys=["hidden"], out_keys=["state_value"]
)

actor_module = TensorDictSequential(feature_extractor, actor_head, extractor)
value_module = TensorDictSequential(feature_extractor, value_head)

policy_module = ProbabilisticActor(
    module=actor_module,
    in_keys=["loc", "scale"],
    spec=transformed_env.action_spec,
    distribution_class=TanhNormal,
    return_log_prob=True,
)


In [ ]:
collector = SyncDataCollector(
    transformed_env,
    policy_module,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames,
    split_trajs=False,
    device=device
)

replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=frames_per_batch),
    sampler=SamplerWithoutReplacement(),
)

advantage_module = GAE(
    gamma=gamma, lmbda=lmbda, value_network=value_module, average_gae=True
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=clip_epsilon,
    entropy_bonus=bool(entropy_eps),
    entropy_coeff=entropy_eps,
    critic_coeff=1.0,
    loss_critic_type="smooth_l1",
)

optim = torch.optim.Adam(loss_module.parameters(), lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, total_frames // frames_per_batch, 0.0
)

In [ ]:
logs = defaultdict(list)
pbar = tqdm(total=total_frames)

for i, tensordict_data in enumerate(collector):

    tensordict_data = tensordict_data.reshape(-1, *tensordict_data.shape[2:])

    for _ in range(num_epochs):
        with torch.no_grad():
            advantage_module(tensordict_data.squeeze(0))

        data_view = tensordict_data.reshape(-1)
        replay_buffer.extend(data_view.cpu())

        # === PPO inner loop ===
        for _ in range(frames_per_batch // sub_batch_size):
            subdata = replay_buffer.sample(sub_batch_size)

            loss_vals = loss_module(subdata.to(device))

            loss = (
                loss_vals["loss_objective"]
                + loss_vals["loss_critic"]
                + loss_vals["loss_entropy"]
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_grad_norm)

            optim.step()
            optim.zero_grad()

    # === logging ===
    logs["reward"].append(tensordict_data["next", "reward"].mean().item())
    pbar.update(tensordict_data.numel())

    # === evaluation ===
    if i % 10 == 0:
        with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
            eval_rollout = transformed_env.rollout(200, policy_module)
            logs["eval_reward"].append(eval_rollout["next", "reward"].mean().item())
            del eval_rollout

    scheduler.step()


In [ ]:
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(logs["reward"])
plt.title("training rewards (average)")
plt.subplot(2, 2, 2)
plt.plot(logs["step_count"])
plt.title("Max step count (training)")
plt.subplot(2, 2, 3)
plt.plot(logs["eval reward (sum)"])
plt.title("Return (test)")
plt.subplot(2, 2, 4)
plt.plot(logs["eval step_count"])
plt.title("Max step count (test)")
plt.show()

In [24]:
torch.save({
    "actor": policy_module.state_dict(),
    "critic": value_module.state_dict(),
    "optimizer": optim.state_dict(),
}, "checkpoint.pt")

### Инференс

In [ ]:
checkpoint = torch.load("checkpoint.pt", map_location=device)

policy_module.load_state_dict(checkpoint["actor"])
policy_module.eval()

In [ ]:
env = VecEnv(n_envs=1, device=device, show_viewer=True)
transformed_env = TransformedEnv(
    env,
    Resize(320, 320, in_keys="observation", out_keys="observation", interpolation="bilinear")
)

In [ ]:
td = transformed_env.reset()

while True:
    with torch.no_grad():
        td = actor_module(td)

    td["action"] = torch.tanh(td["loc"])

    td = transformed_env.step(td)
    td = td["next"]

    if td["done"].any():
        break